# Silver Layer — CRM Customers (`crm_cust_info` → `crm_customers`)

Cleans and transforms the bronze CRM customer data and loads it into `operation.silver.crm_customers`.

**Template note:** This notebook is the template for all other Silver notebooks (`silver_<source>_<table_name>`).
To clone it for another bronze table: duplicate the notebook, update `SOURCE_TABLE` / `TARGET_TABLE` in the Setup cell,
then adapt the data-quality checks and transformation steps to that table's columns.

## Section 1 — Read data from the Bronze table into a DataFrame

In [0]:
# Setup & configuration
# >>> EDIT THESE TWO VARIABLES WHEN CLONING THIS NOTEBOOK AS A TEMPLATE <<<
SOURCE_TABLE = 'operation.bronze.crm_cust_info'   # bronze source table
TARGET_TABLE = 'operation.silver.crm_customers'   # silver target table (friendly name)

from pyspark.sql import functions as F, Window as W

# Read the bronze table into a DataFrame (spark session is provided by the notebook runtime)
df_bronze = spark.table(SOURCE_TABLE)

print(f"Loaded {SOURCE_TABLE}: {df_bronze.count():,} rows")
df_bronze.printSchema()
display(df_bronze.limit(10))

## Section 2 — Analyze data quality using SQL (list all identified issues)

Issues identified in `operation.bronze.crm_cust_info` (18,494 rows, **all columns stored as strings**):

| # | Issue | Finding |
| - | ----- | ------- |
| 1 | Duplicates | 5 duplicate groups on `cst_key`, 6 duplicate groups on `cst_id` (business key) |
| 2 | Nulls | `cst_id`: 4, `cst_firstname`: 8, `cst_lastname`: 7, `cst_marital_status`: 7, **`cst_gndr`: 4,578 (~25%)**, `cst_create_date`: 4 |
| 3 | Extra spaces | leading/trailing whitespace in 17 firstnames and 22 lastnames |
| 4 | Abbreviations | marital status `M`/`S`, gender `M`/`F` need expanding to full words |
| 5 | Date validation | `cst_create_date` is a string in `yyyy-MM-dd` format (range 2025-10-06 → 2026-01-27) and needs casting to a real `date` type |
| 6 | Column names | `cst_*` prefixes are unfriendly → rename (e.g. `cst_id` → `customer_id`) |

The cells below re-verify each finding against the live data.

In [0]:
# DQ check 1 — duplicates on the business key (cst_id) and the surrogate key (cst_key), shown separately
dup_cst_id = df_bronze.groupBy('cst_id').count().filter(F.col('count') > 1)
dup_cst_key = df_bronze.groupBy('cst_key').count().filter(F.col('count') > 1)

print(f"Duplicate groups on cst_id : {dup_cst_id.count()}")
print(f"Duplicate groups on cst_key: {dup_cst_key.count()}")

# Inspect the duplicated cst_id rows (which record should survive the dedup?)
display(df_bronze.join(dup_cst_id.select('cst_id'), on='cst_id', how='inner')
        .orderBy('cst_id', 'cst_create_date'))

In [0]:
# DQ check 2 — nulls per column
null_counts = df_bronze.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df_bronze.columns])
display(null_counts)

# DQ check 3 — leading/trailing whitespace in string columns
for c in ['cst_id', 'cst_key', 'cst_firstname', 'cst_lastname']:
    n = df_bronze.filter(F.col(c) != F.trim(F.col(c))).count()
    print(f'Found {n} rows with leading or trailing whitespace in column {c}')

# DQ check 4 — distinct categorical values (what needs normalizing?)
display(df_bronze.select('cst_marital_status').distinct())
display(df_bronze.select('cst_gndr').distinct())

# DQ check 5 — date format & range: is the string parseable as yyyy-MM-dd, and is the range plausible?
date_range = df_bronze.agg(
    F.min('cst_create_date').alias('min'),
    F.max('cst_create_date').alias('max')).first().asDict()
print('create_date string range:', date_range)
unparseable = df_bronze.filter(
    F.col('cst_create_date').isNotNull()
    & F.to_date(F.col('cst_create_date'), 'yyyy-MM-dd').isNull())
print(f"create_date values not parseable as yyyy-MM-dd: {unparseable.count()}")

### Section 2.1 — Transform data (fix issues one by one)

Each step fixes **one** issue and displays the result before moving to the next.

In [0]:
# Step 1 — trim strings, expand abbreviations (M/S, M/F), fill missing gender/marital values with 'n/a'
df_clean = (df_bronze
    .withColumn('cst_id',        F.trim(F.col('cst_id')))
    .withColumn('cst_key',       F.trim(F.col('cst_key')))
    .withColumn('cst_firstname', F.trim(F.col('cst_firstname')))
    .withColumn('cst_lastname',  F.trim(F.col('cst_lastname')))
    .withColumn('marital_status', F.when(F.col('cst_marital_status') == 'M', 'Married')
                                    .when(F.col('cst_marital_status') == 'S', 'Single')
                                    .otherwise('n/a'))
    .withColumn('gender',         F.when(F.col('cst_gndr') == 'M', 'Male')
                                    .when(F.col('cst_gndr') == 'F', 'Female')
                                    .otherwise('n/a')))

# Verify the normalization result
display(df_clean.groupBy('marital_status', 'gender').count())

In [0]:
# Step 2 — cast create_date from string (yyyy-MM-dd) to a proper date type
df_clean = df_clean.withColumn(
    'create_date', F.to_date(F.col('cst_create_date'), 'yyyy-MM-dd'))

print('New column type: ', df_clean.schema['create_date'].dataType)
display(df_clean.select('cst_id', 'create_date').orderBy(F.col('create_date').desc()))

In [0]:
# Step 3 — drop rows with a missing business key, then deduplicate on cst_id
#          keeping the record with the most recent create_date
df_clean = df_clean.filter(
    F.col('cst_id').isNotNull() & (F.col('cst_id') != ''))

w = W.partitionBy('cst_id').orderBy(F.col('create_date').desc())
df_dedup = (df_clean
    .withColumn('_rn', F.row_number().over(w))
    .filter(F.col('_rn') == 1)
    .drop('_rn'))

print(f"Rows before dedup: {df_clean.count():,} | after dedup: {df_dedup.count():,}")

In [0]:
# Step 4 — rename columns to friendly names for the silver layer
df_silver = (df_dedup
    .select(
        F.col('cst_id').alias('customer_id'),
        F.col('cst_key').alias('customer_key'),
        F.col('cst_firstname').alias('firstname'),
        F.col('cst_lastname').alias('lastname'),
        F.col('marital_status'),
        F.col('gender'),
        F.col('create_date')))

display(df_silver)

### Sanity checks — final DataFrame before writing

Fails loudly (assert) if any check does not hold, so a broken dataset never reaches the silver layer.

In [0]:
# Sanity checks on the final DataFrame before writing
EXPECTED_VALUES = {
    'marital_status': ['Married', 'Single', 'n/a'],
    'gender': ['Male', 'Female', 'n/a'],
}

row_count = df_silver.count()
assert row_count > 0, 'Silver DataFrame is empty'

dup_id = df_silver.groupBy('customer_id').count().filter('count > 1').count()
assert dup_id == 0, f'{dup_id} duplicate customer_id values remain'

dup_key = df_silver.groupBy('customer_key').count().filter('count > 1').count()
assert dup_key == 0, f'{dup_key} duplicate customer_key values remain'

null_ids = df_silver.filter(
    F.col('customer_id').isNull() | (F.col('customer_id') == '')).count()
assert null_ids == 0, 'null/empty customer_id values remain'

for col_name, allowed in EXPECTED_VALUES.items():
    bad = df_silver.filter(~F.col(col_name).isin(allowed)).count()
    assert bad == 0, f"unexpected values in '{col_name}'"

bad_dates = df_silver.filter(
    F.col('create_date').isNotNull()
    & ~F.col('create_date').between(
        F.lit('1900-01-01').cast('date'), F.current_date())).count()
assert bad_dates == 0, f'{bad_dates} implausible create_date values'

print(f'All sanity checks passed  ({row_count:,} rows)')
display(df_silver.limit(20))

## Section 3 — Write the DataFrame to a new Silver table (friendly name)

In [0]:
# Overwrite mode is safe here: this notebook fully recomputes the silver table from bronze.
# overwriteSchema=true replaces the target table's schema — needed when the cleaned schema
# (e.g. create_date cast to DATE) differs from what an earlier run wrote (e.g. create_date as string).
df_silver.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TARGET_TABLE)

# Document what this table contains (lineage documentation for free)
spark.sql(f"""COMMENT ON TABLE {TARGET_TABLE} IS
  'Silver customers: cleaned crm_cust_info from bronze (trimmed strings, normalized marital status/gender,
   create_date cast to date, deduped on customer_id keeping latest create_date)'""")

print(f'Wrote {df_silver.count():,} rows to {TARGET_TABLE}')

### Sanity checks — silver table after writing

Reads the table back from Unity Catalog and validates it independently of the DataFrame above.

In [0]:
# Sanity checks on the written silver table — read back from Unity Catalog
df_check = spark.table(TARGET_TABLE)
bronze_rows = df_bronze.count()
silver_rows = df_check.count()

assert silver_rows > 0 and silver_rows <= bronze_rows, \
    f'unexpected row count: bronze {bronze_rows:,} -> silver {silver_rows:,}'

assert df_check.groupBy('customer_id').count().filter('count > 1').count() == 0, \
    'duplicates on customer_id in silver table'
assert df_check.groupBy('customer_key').count().filter('count > 1').count() == 0, \
    'duplicates on customer_key in silver table'
assert df_check.filter(F.col('customer_id').isNull()).count() == 0, \
    'null customer_id in silver table'

print(f'bronze: {bronze_rows:,} rows -> silver: {silver_rows:,} rows '
      f'(removed {bronze_rows - silver_rows:,} null-key and duplicate rows)')

# Distribution of the normalized categorical values
display(df_check.groupBy('marital_status', 'gender').count())
display(df_check)

## Finalize

* [x] Run the full notebook end to end (Run All) — every cell must succeed top to bottom
* [x] Review structure and readability — one transformation per cell, `display()` after every step
* [x] Add comments and documentation — DQ findings table, step comments, silver table comment
* [ ] **Clone this notebook as a template** for the remaining bronze tables:
  * `silver_crm_prd_info`
  * `silver_crm_sales_details`
  * `silver_erp_cust_az12`
  * `silver_erp_loc_a101`
  * `silver_erp_px_cat_g1v2`

When cloning: update `SOURCE_TABLE` / `TARGET_TABLE` in the Setup cell, replace the DQ findings table with that table's findings, and adapt the transformation steps to its columns.